### Structured Output

In [1]:
import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = ChatGroq(model='openai/gpt-oss-120b')
model.invoke("what is Generative-AI?")

AIMessage(content='**Generative AI** (short for *generative artificial intelligence*) is a class of machine‑learning systems that can **create new content**—such as text, images, audio, video, code, or even molecular structures—rather than just classifying or predicting existing data.\n\n---\n\n## How It Works (in a nutshell)\n\n1. **Training on Large Datasets**  \n   - The model is fed massive amounts of example data (e.g., billions of words, millions of photos).  \n   - It learns statistical patterns, relationships, and the “rules” that govern the data.\n\n2. **Learning a Probability Distribution**  \n   - Instead of learning a single answer, the model learns *how likely* each possible output is given an input.  \n   - This distribution lets the model *sample* new, previously unseen outputs.\n\n3. **Sampling / Decoding**  \n   - When you ask the model to generate something (e.g., “write a poem”), it draws from the learned distribution, producing a novel piece that matches the style a

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The Title of the Movie")
    year:int=Field(description="which year Movie was Released")
    director:str=Field(description="the Movie director name")
    rating:float=Field(description="The Movie rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000280580B2A50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000280580B3770>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The Ti

In [4]:
model.invoke("about RRR Movie")

AIMessage(content='**RRR (2022) – Quick Reference Guide**\n\n| Category | Details |\n|----------|---------|\n| **Full Title** | **Roudram Ranam Rudhiram** (often shortened to **RRR**) |\n| **Release Year** | 2022 (India) |\n| **Language(s)** | Primarily Telugu (original); dubbed in Hindi, Tamil, Malayalam, Kannada, and several international languages |\n| **Genre** | Epic action‑drama, historical‑fiction, period‑fantasy |\n| **Running Time** | 182\u202fminutes (≈\u202f3\u202fhours\u202f2\u202fminutes) |\n| **Director / Writer** | **S.\u202fS.\u202fRajamouli** |\n| **Producers** | D.\u202fV.\u202fVijayendra Prasad (via DV\u202fEntertainments) – co‑produced with *Rashmika Mandanna*, *Mohan Babu*, *Allu Aravind* and *Nithin Reddy* |\n| **Music Composer** | **M.\u202fM.\u202fKreem** (M.\u202fM.\u202fKeeravani) – also responsible for background score |\n| **Cinematography** | **K.\u202fK.\u202fSenthil Kumar** |\n| **Editing** | **A.\u202fSreekar Prasad** |\n| **Production Companies** | DV\u

In [5]:
response = model_with_structure.invoke("about RRR Movie")
response

Movie(title='RRR', year=2022, director='S. S. Rajamouli', rating=8.5)

### Message Output alongside parsed structure

In [10]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke("Give me the title, release year, director, and rating of the movie RRR.")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User asks for title, release year, director, and rating of movie RRR. We need to retrieve via function Movie. Provide those fields. We'll call function.", 'tool_calls': [{'id': 'fc_7e116a7f-c29f-4247-bd3e-8c1b2c132a10', 'function': {'arguments': '{"director":"S. S. Rajamouli","rating":8.5,"title":"RRR","year":2022}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 170, 'total_tokens': 261, 'completion_time': 0.193570487, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.009153765, 'prompt_tokens_details': None, 'queue_time': 0.317878676, 'total_time': 0.202724252}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_49bfac06f1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1109f-ff93-70d0-b8b4-a27815d7f2c5-0', tool_calls=[{'name': 'Movie', '

### Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    age:int

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget in core Rupees")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("give deatails about RRR Movie")
response

MovieDetails(title='RRR', year=2022, cast=[Actor(name='N. T. Rama Rao Jr.', role='Komaram Bheem', age=39), Actor(name='Ram Charan', role='Alluri Sitarama Raju', age=37), Actor(name='Alia Bhatt', role='Sita', age=29), Actor(name='Ajay Devgn', role='Sitaram', age=53)], genres=['Action', 'Drama', 'Historical', 'Adventure'], budget=66000000.0)

### TypedDict

In [12]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie RRR")
response

{'director': 'S. S. Rajamouli', 'rating': 8.5, 'title': 'RRR', 'year': 2022}

In [13]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie RRR")
response

{'budget': 72000000,
 'cast': [{'name': 'N. T. Rama Rao Jr.', 'role': 'Komaram Bheem'},
  {'name': 'Ram Charan', 'role': 'Alluri Sitarama Raju'},
  {'name': 'Alia Bhatt', 'role': 'Sita'},
  {'name': 'Ajay Devgn', 'role': 'Jagannath'},
  {'name': 'Olivia Morris', 'role': 'Jennifer'},
  {'name': 'Sri Karthikeya', 'role': 'Ramesh'},
  {'name': 'Alankar Rao', 'role': "Bheem's friend"},
  {'name': 'Ali', 'role': 'Comedic side character'},
  {'name': 'Shriya Saran', 'role': 'Special appearance'}],
 'genres': ['Action', 'Drama', 'Historical', 'Adventure'],
 'title': 'RRR',
 'year': 2022}

In [14]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [15]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [16]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role":"user","content":"information siva,sivavpr15@gmail.com,984805292"}]
})
result

{'messages': [HumanMessage(content='information siva,sivavpr15@gmail.com,984805292', additional_kwargs={}, response_metadata={}, id='64b2ce0f-e519-44bd-a6ec-dfd0ae2f0ae9'),
  AIMessage(content='{"name":"siva","email":"sivavpr15@gmail.com","phone":"984805292"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 931, 'prompt_tokens': 198, 'total_tokens': 1129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EVwYh50M0cfpanp8UgEoskBIqlCT0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a110b9-3844-7542-9dc2-fcffbed48ae3-0', tool_calls=[], inv

In [17]:
result["structured_response"]

ContactInfo(name='siva', email='sivavpr15@gmail.com', phone='984805292')